<a href="https://colab.research.google.com/github/5ahar-K/CodeGraph-agent/blob/main/1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install networkx anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00


In [1]:
!git clone https://github.com/pallets/click.git target_repo

Cloning into 'target_repo'...
remote: Enumerating objects: 15389, done.
remote: Counting objects: 100% (479/479), done.
remote: Compressing objects: 100% (239/239), done.
remote: Total 15389 (delta 402), reused 245 (delta 238), pack-reused 14910 (from 3)
Receiving objects: 100% (15389/15389), 5.42 MiB | 11.84 MiB/s, done.
Resolving deltas: 100% (10502/10502), done.


In [4]:
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

In [5]:
import ast

with open("target_repo/src/click/core.py") as f:
    tree = ast.parse(f.read())

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        print(f"Found function: {node.name} at line {node.lineno}")

Found function: _complete_visible_commands at line 63
Found function: _check_nested_chain at line 82
Found function: _format_deprecated_label at line 102
Found function: _format_deprecated_suffix at line 110
Found function: batch at line 119
Found function: augment_usage_errors at line 124
Found function: iter_params_for_processing at line 142
Found function: _check_iter at line 2170
Found function: __getattr__ at line 3771
Found function: sort_key at line 158
Found function: __init__ at line 340
Found function: protected_args at line 517
Found function: to_info_dict at line 528
Found function: __enter__ at line 549
Found function: __exit__ at line 554
Found function: scope at line 569
Found function: meta at line 607
Found function: make_formatter at line 634
Found function: with_resource at line 648
Found function: call_on_close at line 677
Found function: close at line 689
Found function: _close_with_exception_info at line 696
Found function: command_path at line 715
Found function:

In [33]:
import ast
import os
import networkx as nx

def find_function_calls(func_node):
    #Given a function's node, return names of functions/methods it calls
    calls = []
    for node in ast.walk(func_node):
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                calls.append(node.func.id) # Simple call, e.g. helper_function()
            elif isinstance(node.func, ast.Attribute): # Method call, e.g. obj.method()
                calls.append(node.func.attr)
    return calls

def build_graph(repo_path):
    graph = nx.DiGraph()
    for root, _, files in os.walk(repo_path):   #root= current folder we are in
        for filename in files:
            if not filename.endswith(".py"):
                continue
            filepath = os.path.join(root, filename)
            try:
                with open(filepath, encoding="utf-8") as f:
                    tree = ast.parse(f.read(), filename=filepath)
            except SyntaxError:
                continue  # skip files that fail to parse
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    graph.add_node(node.name, file=filepath, line=node.lineno)
                    for called in find_function_calls(node):
                        graph.add_edge(node.name, called)  #adding edges for every called function no matter simple or method call
    return graph

In [7]:
g = build_graph("target_repo")
print(f"Found {g.number_of_nodes()} functions, {g.number_of_edges()} call relationships")

Found 1426 functions, 4944 call relationships


In [8]:
def what_calls(graph, function_name):   #Who calls this function
    return list(graph.predecessors(function_name))

def what_does_it_call(graph, function_name):
    return list(graph.successors(function_name))

In [9]:
print(what_calls(g, "echo"))
print(what_does_it_call(g, "echo"))

['ship_new', 'ship_move', 'ship_shoot', 'mine_set', 'mine_remove', 'clone', 'delete', 'setuser', 'commit', 'copy', 'set_config', 'cli', 'log', 'push', 'pull', 'status', 'alias', 'open_cmd', 'save_cmd', 'display_cmd', 'resize_cmd', 'crop_cmd', 'transpose_cmd', 'blur_cmd', 'smoothen_cmd', 'emboss_cmd', 'sharpen_cmd', 'paste_cmd', 'colordemo', 'edit', 'menu', 'ls', 'show_env', 'select_user', 'show', 'prompt', 'confirm', 'clear', 'secho', 'pause', 'prompt_func', 'invoke', 'main', 'handle_parse_result', 'shell_complete', '_check_version', 'version_option', 'custom_version_option', 'help_option', 'callback', 'show_version', 'show_help', 'render_progress', 'test_other_command_invoke', 'test_other_command_forward', 'test_forwarded_params_consistency', 'test_default_maps', 'test_group_with_args', 'test_custom_parser', 'test_object_propagation', 'test_invoked_subcommand', 'test_aliased_command_canonical_name', 'test_help_param_priority', 'test_unprocessed_options', 'other_cmd', 'test', 'first', 

In [10]:
!pip install -q google-genai

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [11]:
def get_function_source(graph, function_name):
    """Pull the actual source code of a function using its stored file location."""
    node_data = graph.nodes[function_name]
    with open(node_data["file"]) as f:
        source_text = f.read()
    tree = ast.parse(source_text)
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            return ast.get_source_segment(source_text, node)
    return None

def ask_about_function(graph, function_name, question):
    source = get_function_source(graph, function_name)
    callers = what_calls(graph, function_name)
    callees = what_does_it_call(graph, function_name)

    prompt = f"""Here is a function called `{function_name}`:

{source}

It is called by: {callers}
It calls: {callees}

Question: {question}

Answer using only the information given above. If you can't determine the answer from this, say so."""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return response.text

In [12]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [13]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Hello, just testing"
)
print(response.text)

Hello! Test successful—everything is working properly on my end. 

How can I help you today?


In [14]:
print(ask_about_function(g, "echo", "What other functions call this one?"))

Based on the provided information, `echo` is called by the following functions:

* `ship_new`
* `ship_move`
* `ship_shoot`
* `mine_set`
* `mine_remove`
* `clone`
* `delete`
* `setuser`
* `commit`
* `copy`
* `set_config`
* `cli`
* `log`
* `push`
* `pull`
* `status`
* `alias`
* `open_cmd`
* `save_cmd`
* `display_cmd`
* `resize_cmd`
* `crop_cmd`
* `transpose_cmd`
* `blur_cmd`
* `smoothen_cmd`
* `emboss_cmd`
* `sharpen_cmd`
* `paste_cmd`
* `colordemo`
* `edit`
* `menu`
* `ls`
* `show_env`
* `select_user`
* `show`
* `prompt`
* `confirm`
* `clear`
* `secho`
* `pause`
* `prompt_func`
* `invoke`
* `main`
* `handle_parse_result`
* `shell_complete`
* `_check_version`
* `version_option`
* `custom_version_option`
* `help_option`
* `callback`
* `show_version`
* `show_help`
* `render_progress`
* `test_other_command_invoke`
* `test_other_command_forward`
* `test_forwarded_params_consistency`
* `test_default_maps`
* `test_group_with_args`
* `test_custom_parser`
* `test_object_propagation`
* `test_invo

In [15]:
answer = ask_about_function(g, "echo", "What would break if this function's return type changed?")
print(answer)

Based on the provided information, it cannot be determined what would break if the function's return type changed. The prompt lists the callers and called functions, but does not provide details about how callers use the return value of `echo`.


In [18]:
def generate_test(graph, function_name):
    source = get_function_source(graph, function_name)
    prompt = f"""Write a pytest unit test for this function:

{source}

Only output the test code, no explanation."""
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    return response.text


print(generate_test(g, "echo"))

```python
from unittest.mock import call, patch
import pytest


def test_echo():
    with (
        patch("click.prompt", side_effect=[1, 2, 3]) as mock_prompt,
        patch("click.echo") as mock_echo,
    ):
        echo()

        assert mock_prompt.call_args_list == [call("", type=int)] * 3
        assert mock_echo.call_args_list == [call(1), call(2), call(3)]
```


In [19]:
print(get_function_source(g, "echo"))

def echo():
        for _ in range(3):
            click.echo(click.prompt("", type=int))


In [28]:
import click
from unittest.mock import call, patch
import pytest


def echo():
    for _ in range(3):
        click.echo(click.prompt("", type=int))


def test_echo():
    with (
        patch("click.prompt", side_effect=[1, 2, 3]) as mock_prompt,
        patch("click.echo") as mock_echo,
    ):
        echo()

        assert mock_prompt.call_args_list == [call("", type=int)] * 3
        assert mock_echo.call_args_list == [call(1), call(2), call(3)]


test_echo()

In [30]:
print(g.nodes["echo"])

{'file': 'target_repo/tests/test_utils/test_prompt.py', 'line': 94}
